In [ ]:
!pip install pandas numpy scikit-learn matplotlib seaborn shap lime xgboost

In [ ]:
import warnings
# Only suppress specific known non-actionable warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message='.*lbfgs.*')

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_curve, auc, precision_score, recall_score, f1_score)
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance
from xgboost import XGBClassifier
import shap
import lime
import lime.lime_tabular
import joblib
import os

In [ ]:
import os
import pandas as pd

INPUT_DIR = "/kaggle/input"

# Walk through Kaggle input folders
for dirname, _, filenames in os.walk(INPUT_DIR):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# Set dataset path manually after checking output
DATA_PATH = "/kaggle/input/datasets/nabihazahid/loan-prediction-dataset-2025/loan_dataset_20000.csv"

data = pd.read_csv(DATA_PATH)

print(data.shape)

In [ ]:
data.head()

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
print(data.columns.tolist())

In [ ]:
# ============================================
# CLASS BALANCE CHECK
# ============================================

print("=== Target Variable Distribution ===")

target_col = 'loan_paid_back'

print(data[target_col].value_counts())
print()

print(data[target_col].value_counts(normalize=True).round(4) * 100)

plt.figure(figsize=(6, 4))

data[target_col].value_counts().plot(
    kind='bar',
    color=['#2ecc71', '#e74c3c']
)

plt.title('Loan Paid Back Distribution')
plt.xlabel('Class')
plt.ylabel('Count')

plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

majority_pct = data[target_col].value_counts(normalize=True).max()

if majority_pct > 0.7:
    print(f"WARNING: Dataset is imbalanced ({majority_pct:.1%} majority class)")
    print("Consider using stratified splitting, SMOTE, or class_weight='balanced'")
else:
    print("Class distribution is reasonably balanced")

In [ ]:
# ============================================
# DATA QUALITY: NEGATIVE VALUES
# ============================================
print("=== Checking for Negative Values in Asset Columns ===")
asset_cols = ['residential_assets_value', 'commercial_assets_value',
              'luxury_assets_value', 'bank_asset_value']

asset_cols_actual = []
for col in asset_cols:
    if col in data.columns:
        asset_cols_actual.append(col)
    elif ' ' + col in data.columns:
        asset_cols_actual.append(' ' + col)

for col in asset_cols_actual:
    neg_count = (data[col] < 0).sum()
    if neg_count > 0:
        print(f"WARNING: {col.strip()}: {neg_count} negative values (min={data[col].min():,.0f})")
    else:
        print(f"OK: {col.strip()}: no negative values")

# ============================================
# OUTLIER ANALYSIS (IQR Method)
# ============================================
print("\n=== Outlier Detection (IQR Method) ===")
numeric_cols = data.select_dtypes(include=[np.number]).columns.tolist()
# Handle possible leading space in column name (M3 fix)
numeric_cols = [c for c in numeric_cols if c.strip() != 'loan_id']

for col in numeric_cols:
    Q1 = data[col].quantile(0.25)
    Q3 = data[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_outliers = ((data[col] < lower) | (data[col] > upper)).sum()
    pct = n_outliers / len(data) * 100
    if n_outliers > 0:
        print(f"  {col.strip()}: {n_outliers} outliers ({pct:.1f}%)")

# Boxplots
n_cols_plot = min(4, len(numeric_cols))
n_rows_plot = (len(numeric_cols) + n_cols_plot - 1) // n_cols_plot
fig, axes = plt.subplots(n_rows_plot, n_cols_plot, figsize=(16, 3 * n_rows_plot))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]
for i, col in enumerate(numeric_cols):
    if i < len(axes_flat):
        data.boxplot(column=col, ax=axes_flat[i])
        axes_flat[i].set_title(col.strip(), fontsize=8)
for j in range(len(numeric_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)
plt.suptitle('Boxplot Outlier Analysis', fontsize=14)
plt.tight_layout()
plt.show()

# ============================================
# DUPLICATE ROW DETECTION
# ============================================
n_dupes = data.duplicated().sum()
if n_dupes > 0:
    print(f"\nWARNING: {n_dupes} duplicate rows detected ({n_dupes/len(data)*100:.1f}%)")
    print("Consider: data = data.drop_duplicates()")
else:
    print(f"\nOK: No duplicate rows detected")

In [ ]:
# ============================================
# VISUALIZATIONS
# ============================================

target_col = 'loan_paid_back'

# 1. Correlation Heatmap
plt.figure(figsize=(12, 8))

numeric_data = data.select_dtypes(include=['int64', 'float64'])
corr_matrix = numeric_data.corr()

sns.heatmap(
    corr_matrix,
    annot=True,
    cmap='coolwarm',
    fmt=".2f",
    linewidths=0.5
)

plt.title('Feature Correlation Heatmap')
plt.show()

# 2. Distribution of Loan Status
plt.figure(figsize=(6, 4))

sns.countplot(
    x=target_col,
    data=data,
    palette='Set2'
)

plt.title('Distribution of Loan Paid Back')
plt.xlabel('Loan Paid Back')
plt.ylabel('Count')

plt.show()

In [ ]:
# ============================================
# FEATURE DISTRIBUTION BY CLASS
# ============================================

target_col = 'loan_paid_back'

# Select numeric columns except target
plot_features = [
    col for col in data.select_dtypes(include=[np.number]).columns
    if col != target_col
]

# Get class labels dynamically
class_labels = sorted(data[target_col].unique())

# Class names + colors
class_display = []

for lbl in class_labels:
    
    if str(lbl) == '0':
        class_display.append((lbl, '#e74c3c', 'Not Paid Back'))
    else:
        class_display.append((lbl, '#2ecc71', 'Paid Back'))

# Grid setup
n_feats = len(plot_features)

n_cols = 3
n_rows = (n_feats + n_cols - 1) // n_cols

fig, axes = plt.subplots(
    n_rows,
    n_cols,
    figsize=(18, 4 * n_rows)
)

axes = axes.flatten()

# Plot feature distributions
for i, col in enumerate(plot_features):

    ax = axes[i]

    for label, color, name in class_display:

        subset = data[data[target_col] == label][col]

        ax.hist(
            subset,
            bins=30,
            alpha=0.5,
            label=name,
            color=color,
            density=True
        )

    ax.set_title(col, fontsize=10, fontweight='bold')
    ax.legend(fontsize=8)

# Hide empty subplots
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle(
    'Feature Distribution by Loan Repayment Status',
    fontsize=16,
    fontweight='bold'
)

plt.tight_layout()

plt.show()

In [ ]:
data.isnull().sum()

In [ ]:
# Verify data shape before preprocessing
print(f"Shape: {data.shape}")
print(f"Columns: {list(data.columns)}")
data.head(3)

In [ ]:
# Column name cleanup
data.columns = data.columns.str.strip()

In [ ]:
# ============================================
# FEATURE ENGINEERING
# ============================================

# Income to loan ratio
data['income_loan_ratio'] = (
    data['annual_income'] / (data['loan_amount'] + 1)
)

# Credit utilization ratio
data['credit_utilization_ratio'] = (
    data['current_balance'] / (data['total_credit_limit'] + 1)
)

# Installment to income ratio
data['installment_income_ratio'] = (
    data['installment'] / (data['monthly_income'] + 1)
)

# Available credit
data['available_credit'] = (
    data['total_credit_limit'] - data['current_balance']
)

# Display new features
print(data[[
    'income_loan_ratio',
    'credit_utilization_ratio',
    'installment_income_ratio',
    'available_credit'
]].head())

In [ ]:
# ============================================
# LABEL ENCODING
# ============================================

from sklearn.preprocessing import LabelEncoder

# Create encoders dictionary
label_encoders = {}

# Categorical columns
categorical_cols = [
    'gender',
    'marital_status',
    'education_level',
    'employment_status',
    'loan_purpose',
    'grade_subgrade'
]

# Encode categorical features
for col in categorical_cols:
    
    le = LabelEncoder()
    
    data[col] = le.fit_transform(data[col].astype(str))
    
    label_encoders[col] = le

# Target encoding
# loan_paid_back:
# 1 = Paid Back
# 0 = Not Paid Back

data['loan_paid_back'] = data['loan_paid_back'].astype(int)

# Check results
print("Encoding complete\n")

for col in categorical_cols:
    print(f"{col}: {data[col].nunique()} unique encoded values")

print("\nTarget values:")
print(sorted(data['loan_paid_back'].unique()))

In [ ]:
# Remove unnecessary identifier columns if present
columns_to_drop = ['loan_id', 'customer_id', 'id']

data = data.drop(columns=columns_to_drop, errors='ignore')

In [ ]:
data.head()  # loan id is not there

In [ ]:
# ============================================
# FEATURE / TARGET SPLIT
# ============================================

target_col = 'loan_paid_back'

X = data.drop(target_col, axis=1)
y = data[target_col]

print(f"Features: {list(X.columns)}")
print(f"Feature count: {X.shape[1]}")

print("\nTarget distribution:")
print(y.value_counts())

# ============================================
# STRATIFIED TRAIN/TEST SPLIT
# ============================================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTrain size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")

# ============================================
# STANDARD SCALING
# ============================================

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print("\nScaling applied successfully")

# ============================================
# CROSS-VALIDATION SETUP
# ============================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

print("5-fold stratified cross-validation configured")

# DECISION TREE

In [ ]:
# ============================================
# DECISION TREE — HYPERPARAMETER TUNING
# ============================================
# Note: Tree models are scale-invariant. Scaling has no effect on
# their predictions but is applied uniformly for pipeline simplicity.
dt_param_grid = {
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid=dt_param_grid,
    cv=cv,
    scoring='accuracy',  # Consider 'f1' or 'roc_auc' for imbalanced data
    n_jobs=-1,
    verbose=0
)
dt_grid.fit(X_train_scaled, y_train)
dt_model = dt_grid.best_estimator_  # Refit on full X_train_scaled (refit=True default)

dt_pred = dt_model.predict(X_test_scaled)
dt_prob = dt_model.predict_proba(X_test_scaled)[:, 1]

# Use GridSearchCV's internal CV results (no redundant cross_val_score)
dt_cv_mean = dt_grid.best_score_
dt_cv_std = dt_grid.cv_results_['std_test_score'][dt_grid.best_index_]

print("=== Decision Tree Results ===")
print(f"Best Params:   {dt_grid.best_params_}")
print(f"CV Accuracy:   {dt_cv_mean:.4f} +/- {dt_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, dt_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, dt_pred, target_names=['Rejected', 'Approved']))

In [ ]:
dt_cm = confusion_matrix(y_test, dt_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=dt_cm)
disp.plot(cmap='Blues')
plt.title("Decision Tree Confusion Matrix")
plt.show()

In [ ]:
dt_fpr, dt_tpr, _ = roc_curve(y_test, dt_prob)
dt_roc_auc = auc(dt_fpr, dt_tpr)

plt.figure(figsize=(7, 5))
plt.plot(dt_fpr, dt_tpr, label=f"AUC = {dt_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Decision Tree')
plt.legend()
plt.show()

In [ ]:
dt_importance = dt_model.feature_importances_

feat_imp = pd.Series(dt_importance, index=X.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(10, 6))

plt.title("Feature Importance - Decision Tree")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

# LOGISTIC REGRESSION

In [ ]:
# ============================================
# LOGISTIC REGRESSION — HYPERPARAMETER TUNING
# ============================================
# Use a list of dicts to handle solver-penalty compatibility:
#   - 'lbfgs' supports only 'l2'
#   - 'liblinear' supports both 'l1' and 'l2'
lr_param_grid = [
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l2'], 'solver': ['lbfgs', 'liblinear']},
    {'C': [0.01, 0.1, 1, 10, 100], 'penalty': ['l1'], 'solver': ['liblinear']},
]

lr_grid = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid=lr_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
lr_grid.fit(X_train_scaled, y_train)
lr_model = lr_grid.best_estimator_

lr_pred = lr_model.predict(X_test_scaled)
lr_prob = lr_model.predict_proba(X_test_scaled)[:, 1]

lr_cv_mean = lr_grid.best_score_
lr_cv_std = lr_grid.cv_results_['std_test_score'][lr_grid.best_index_]

print("=== Logistic Regression Results ===")
print(f"Best Params:   {lr_grid.best_params_}")
print(f"CV Accuracy:   {lr_cv_mean:.4f} +/- {lr_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, lr_pred, target_names=['Rejected', 'Approved']))

In [ ]:
lr_cm = confusion_matrix(y_test, lr_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=lr_cm)
disp.plot(cmap='Greens')
plt.title("Logistic Regression Confusion Matrix")
plt.show()

In [ ]:
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_prob)
lr_roc_auc = auc(lr_fpr, lr_tpr)

plt.figure(figsize=(7, 5))
plt.plot(lr_fpr, lr_tpr, label=f"AUC = {lr_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Logistic Regression')
plt.legend()
plt.show()

In [ ]:
lr_importance = lr_model.coef_[0]

feat_imp = pd.Series(lr_importance, index=X.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(10, 6))

plt.title("Feature Importance (Coefficients) - Logistic Regression")
plt.xlabel("Coefficient Value")
plt.ylabel("Features")
plt.show()

# RANDOM FOREST

In [ ]:
# ============================================
# RANDOM FOREST — HYPERPARAMETER TUNING
# ============================================
rf_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2]
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=rf_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
rf_grid.fit(X_train_scaled, y_train)
rf_model = rf_grid.best_estimator_

rf_pred = rf_model.predict(X_test_scaled)
rf_prob = rf_model.predict_proba(X_test_scaled)[:, 1]

rf_cv_mean = rf_grid.best_score_
rf_cv_std = rf_grid.cv_results_['std_test_score'][rf_grid.best_index_]

print("=== Random Forest Results ===")
print(f"Best Params:   {rf_grid.best_params_}")
print(f"CV Accuracy:   {rf_cv_mean:.4f} +/- {rf_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, rf_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, rf_pred, target_names=['Rejected', 'Approved']))

In [ ]:
rf_cm = confusion_matrix(y_test, rf_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=rf_cm)
disp.plot(cmap='Oranges')
plt.title("Random Forest Confusion Matrix")
plt.show()

In [ ]:
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_prob)
rf_roc_auc = auc(rf_fpr, rf_tpr)

plt.figure(figsize=(7, 5))
plt.plot(rf_fpr, rf_tpr, label=f"AUC = {rf_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Random Forest')
plt.legend()
plt.show()

In [ ]:
rf_importance = rf_model.feature_importances_

feat_imp = pd.Series(rf_importance, index=X.columns)
feat_imp.sort_values().plot(kind='barh', figsize=(10, 6))

plt.title("Feature Importance - Random Forest")
plt.xlabel("Importance Score")
plt.ylabel("Features")
plt.show()

# XGBOOST

In [ ]:
# ============================================
# XGBOOST — HYPERPARAMETER TUNING
# ============================================
xgb_param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0]
}

# n_jobs=1 on the estimator avoids thread contention with GridSearchCV's n_jobs=-1 (H3)
xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss', n_jobs=1),
    param_grid=xgb_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
xgb_grid.fit(X_train_scaled, y_train)
xgb_model = xgb_grid.best_estimator_

xgb_pred = xgb_model.predict(X_test_scaled)
xgb_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

xgb_cv_mean = xgb_grid.best_score_
xgb_cv_std = xgb_grid.cv_results_['std_test_score'][xgb_grid.best_index_]

print("=== XGBoost Results ===")
print(f"Best Params:   {xgb_grid.best_params_}")
print(f"CV Accuracy:   {xgb_cv_mean:.4f} +/- {xgb_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, xgb_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, xgb_pred, target_names=['Rejected', 'Approved']))

# Confusion Matrix
xgb_cm = confusion_matrix(y_test, xgb_pred)
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=xgb_cm)
disp.plot(cmap='Blues')
plt.title("XGBoost Confusion Matrix")
plt.show()

# ROC Curve
xgb_fpr, xgb_tpr, _ = roc_curve(y_test, xgb_prob)
xgb_roc_auc = auc(xgb_fpr, xgb_tpr)
plt.figure(figsize=(7, 5))
plt.plot(xgb_fpr, xgb_tpr, label=f"AUC = {xgb_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("XGBoost ROC Curve")
plt.legend()
plt.show()

# Feature Importance
xgb_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(xgb_importance_df['Feature'], xgb_importance_df['Importance'])
plt.title("XGBoost Feature Importance")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.show()
print(xgb_importance_df)

# KNN

In [ ]:
# ============================================
# KNN — HYPERPARAMETER TUNING
# ============================================
knn_param_grid = {
    'n_neighbors': [3, 5, 7, 9, 11],
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan']
}

knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    param_grid=knn_param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)
knn_grid.fit(X_train_scaled, y_train)
knn_model = knn_grid.best_estimator_

knn_pred = knn_model.predict(X_test_scaled)
knn_prob = knn_model.predict_proba(X_test_scaled)[:, 1]

knn_cv_mean = knn_grid.best_score_
knn_cv_std = knn_grid.cv_results_['std_test_score'][knn_grid.best_index_]

print("=== KNN Results ===")
print(f"Best Params:   {knn_grid.best_params_}")
print(f"CV Accuracy:   {knn_cv_mean:.4f} +/- {knn_cv_std:.4f}")
print(f"Test Accuracy: {accuracy_score(y_test, knn_pred):.4f}")
print()
print("Classification Report:")
print(classification_report(y_test, knn_pred, target_names=['Rejected', 'Approved']))

# Confusion Matrix
knn_cm = confusion_matrix(y_test, knn_pred)
plt.figure(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=knn_cm)
disp.plot(cmap='Greens')
plt.title("KNN Confusion Matrix")
plt.show()

# ROC Curve
knn_fpr, knn_tpr, _ = roc_curve(y_test, knn_prob)
knn_roc_auc = auc(knn_fpr, knn_tpr)
plt.figure(figsize=(7, 5))
plt.plot(knn_fpr, knn_tpr, label=f"AUC = {knn_roc_auc:.2f}")
plt.plot([0, 1], [0, 1], linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("KNN ROC Curve")
plt.legend()
plt.show()

# Permutation Importance
perm_importance = permutation_importance(
    knn_model, X_test_scaled, y_test, n_repeats=10, random_state=42
)
knn_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': perm_importance.importances_mean
}).sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(knn_importance_df['Feature'], knn_importance_df['Importance'])
plt.title("KNN Feature Importance (Permutation)")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.show()
print(knn_importance_df)

In [ ]:
# ============================================
# COMPREHENSIVE MODEL COMPARISON
# ============================================

model_info = {
    'Decision Tree':       {'pred': dt_pred,  'prob': dt_prob,  'cv_mean': dt_cv_mean,  'cv_std': dt_cv_std,  'model': dt_model},
    'Logistic Regression': {'pred': lr_pred,  'prob': lr_prob,  'cv_mean': lr_cv_mean,  'cv_std': lr_cv_std,  'model': lr_model},
    'Random Forest':       {'pred': rf_pred,  'prob': rf_prob,  'cv_mean': rf_cv_mean,  'cv_std': rf_cv_std,  'model': rf_model},
    'XGBoost':             {'pred': xgb_pred, 'prob': xgb_prob, 'cv_mean': xgb_cv_mean, 'cv_std': xgb_cv_std, 'model': xgb_model},
    'KNN':                 {'pred': knn_pred, 'prob': knn_prob, 'cv_mean': knn_cv_mean, 'cv_std': knn_cv_std, 'model': knn_model},
}

comparison_rows = []
for name, info in model_info.items():
    acc = accuracy_score(y_test, info['pred'])
    prec = precision_score(y_test, info['pred'], pos_label=1)   # Positive = Approved
    rec = recall_score(y_test, info['pred'], pos_label=1)
    f1 = f1_score(y_test, info['pred'], pos_label=1)
    fpr_vals, tpr_vals, _ = roc_curve(y_test, info['prob'])
    roc = auc(fpr_vals, tpr_vals)
    comparison_rows.append({
        'Model': name,
        'Test Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1 Score': round(f1, 4),
        'AUC-ROC': round(roc, 4),
        'CV Accuracy (mean)': round(info['cv_mean'], 4),
        'CV Accuracy (std)': round(info['cv_std'], 4),
    })

comparison_df = pd.DataFrame(comparison_rows).set_index('Model')
print("Note: Precision/Recall/F1 computed for pos_label=1 (Approved)")
print(comparison_df.to_string())

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sorted_df = comparison_df.sort_values('Test Accuracy')
axes[0].barh(sorted_df.index, sorted_df['Test Accuracy'], color='steelblue')
axes[0].set_xlabel('Accuracy')
axes[0].set_title('Test Accuracy Comparison')
for i, v in enumerate(sorted_df['Test Accuracy']):
    axes[0].text(v + 0.002, i, f'{v:.4f}', va='center')

axes[1].barh(
    comparison_df.index,
    comparison_df['CV Accuracy (mean)'],
    xerr=comparison_df['CV Accuracy (std)'],
    color='coral', capsize=5
)
axes[1].set_xlabel('CV Accuracy')
axes[1].set_title('Cross-Validation Accuracy (5-fold)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# MULTI-MODEL ROC CURVE OVERLAY
# ============================================
plt.figure(figsize=(10, 8))

model_roc_data = {
    'Decision Tree': (dt_prob, '#e74c3c'),
    'Logistic Regression': (lr_prob, '#3498db'),
    'Random Forest': (rf_prob, '#2ecc71'),
    'XGBoost': (xgb_prob, '#f39c12'),
    'KNN': (knn_prob, '#9b59b6'),
}

for name, (prob, color) in model_roc_data.items():
    fpr_vals, tpr_vals, _ = roc_curve(y_test, prob)
    roc_auc_val = auc(fpr_vals, tpr_vals)
    plt.plot(fpr_vals, tpr_vals, label=f"{name} (AUC = {roc_auc_val:.4f})", color=color, linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Baseline')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curve Comparison — All Models', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================
# BEST MODEL SELECTION (using CV accuracy)
# ============================================
best_model_name = comparison_df['CV Accuracy (mean)'].idxmax()
best_model = model_info[best_model_name]['model']

print(f"Best Performing Model: {best_model_name}")
print(f"  Test Accuracy:  {comparison_df.loc[best_model_name, 'Test Accuracy']}")
print(f"  CV Accuracy:    {comparison_df.loc[best_model_name, 'CV Accuracy (mean)']} +/- {comparison_df.loc[best_model_name, 'CV Accuracy (std)']}")
print(f"  F1 Score:       {comparison_df.loc[best_model_name, 'F1 Score']}")
print(f"  AUC-ROC:        {comparison_df.loc[best_model_name, 'AUC-ROC']}")

In [ ]:
# ============================================
# SAVE BEST MODEL + SCALER
# ============================================
MODEL_SAVE_PATH = 'loan_model.pkl'
SCALER_SAVE_PATH = 'scaler.pkl'

joblib.dump(best_model, MODEL_SAVE_PATH)
joblib.dump(scaler, SCALER_SAVE_PATH)
joblib.dump(best_model_name, 'best_model_name.pkl')

print(f"Best model ({best_model_name}) saved to: {MODEL_SAVE_PATH}")
print(f"Scaler saved to: {SCALER_SAVE_PATH}")
print(f"Model name saved to: best_model_name.pkl")

In [ ]:
# ============================================
# SHAP EXPLAINABILITY - GLOBAL + LOCAL
# ============================================

if best_model_name in ['Decision Tree', 'Random Forest', 'XGBoost']:
    shap_explainer = shap.TreeExplainer(best_model)
    shap_values = shap_explainer.shap_values(X_test_scaled)
    if isinstance(shap_values, list):
        shap_values_class1 = shap_values[1]
        base_value = shap_explainer.expected_value[1]
    else:
        shap_values_class1 = shap_values
        base_value = shap_explainer.expected_value
    # Use full test set for display (unscaled for interpretable feature values)
    X_shap_display = X_test
else:
    # KernelExplainer: compute on a subset (slow otherwise)
    shap_explainer = shap.KernelExplainer(best_model.predict_proba, X_train_scaled.iloc[:100])
    shap_values = shap_explainer.shap_values(X_test_scaled.iloc[:50])
    shap_values_class1 = shap_values[1]
    base_value = shap_explainer.expected_value[1]
    # Use the SAME subset for display (unscaled for interpretable feature values)
    X_shap_display = X_test.iloc[:50]

print(f"SHAP explainer created for: {best_model_name}")
print(f"SHAP values shape: {shap_values_class1.shape}, Display data shape: {X_shap_display.shape}")

# Global Feature Importance (bar plot)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_class1, X_shap_display, plot_type='bar', show=False)
plt.title(f'SHAP Feature Importance - {best_model_name}')
plt.tight_layout()
plt.show()

# ============================================
# SHAP BEESWARM (DOT) PLOT
# ============================================
# Shows BOTH feature importance AND feature effect direction per sample
# Color: red = high feature value, blue = low feature value
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values_class1, X_shap_display, show=False)
plt.title(f'SHAP Beeswarm Plot — {best_model_name}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Local Explanation - Waterfall
idx = 0
print(f"\nSample {idx}: Actual={y_test.iloc[idx]}, Predicted={best_model.predict(X_test_scaled.iloc[[idx]])[0]}")

plt.figure(figsize=(12, 8))
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values_class1[idx],
        base_values=base_value,
        data=X_shap_display.iloc[idx],  # Unscaled values for interpretable labels
        feature_names=X.columns.tolist()
    ),
    max_display=13
)
plt.tight_layout()
plt.show()

joblib.dump(shap_explainer, 'shap_explainer.pkl')
print("SHAP explainer saved to: shap_explainer.pkl")

In [ ]:
# ============================================
# LIME EXPLAINABILITY - Local Explanation
# ============================================

lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled.values,
    feature_names=X.columns.tolist(),
    class_names=['Rejected', 'Approved'],
    mode='classification'
)

idx = 0
lime_exp = lime_explainer.explain_instance(
    data_row=X_test_scaled.iloc[idx].values,
    predict_fn=best_model.predict_proba,
    num_features=13
)

print(f"LIME Explanation for Sample {idx}:")
print(f"  Actual: {y_test.iloc[idx]}, Predicted: {best_model.predict(X_test_scaled.iloc[[idx]])[0]}")
print()

fig = lime_exp.as_pyplot_figure()
fig.set_size_inches(12, 6)
plt.title(f'LIME Explanation - {best_model_name} - Sample {idx}')
plt.tight_layout()
plt.show()

print("\nFeature contributions:")
for feat, weight in lime_exp.as_list():
    direction = "-> Approved" if weight > 0 else "-> Rejected"
    print(f"  {feat}: {weight:+.4f} {direction}")

#joblib.dump(lime_explainer, 'lime_explainer.pkl')
print("\nLIME explainer saved to: lime_explainer.pkl")

In [ ]:
import gradio as gr
import pandas as pd
import numpy as np

# ============================================
# PREDICTION FUNCTION
# ============================================

def predict_loan(
    age,
    gender,
    marital_status,
    education_level,
    annual_income,
    monthly_income,
    employment_status,
    debt_to_income_ratio,
    credit_score,
    loan_amount,
    loan_purpose,
    interest_rate,
    loan_term,
    installment,
    grade_subgrade,
    num_of_open_accounts,
    total_credit_limit,
    current_balance,
    delinquency_history,
    public_records,
    num_of_delinquencies
):

    # Create input dataframe
    input_data = pd.DataFrame([{
        'age': age,
        'gender': gender,
        'marital_status': marital_status,
        'education_level': education_level,
        'annual_income': annual_income,
        'monthly_income': monthly_income,
        'employment_status': employment_status,
        'debt_to_income_ratio': debt_to_income_ratio,
        'credit_score': credit_score,
        'loan_amount': loan_amount,
        'loan_purpose': loan_purpose,
        'interest_rate': interest_rate,
        'loan_term': loan_term,
        'installment': installment,
        'grade_subgrade': grade_subgrade,
        'num_of_open_accounts': num_of_open_accounts,
        'total_credit_limit': total_credit_limit,
        'current_balance': current_balance,
        'delinquency_history': delinquency_history,
        'public_records': public_records,
        'num_of_delinquencies': num_of_delinquencies
    }])

    # Encode categorical columns
    for col in categorical_cols:
        input_data[col] = label_encoders[col].transform(input_data[col])

    # Scale
    input_scaled = scaler.transform(input_data)

    # Predict
    prediction = best_model.predict(input_scaled)[0]
    probability = best_model.predict_proba(input_scaled)[0][1]

    if prediction == 1:
        result = "LOAN PAID BACK"
    else:
        result = "LOAN NOT PAID BACK"

    return f"""
Prediction: {result}

Probability of Repayment: {probability:.2%}
"""

# ============================================
# GRADIO INTERFACE
# ============================================

demo = gr.Interface(
    fn=predict_loan,

    inputs=[

        gr.Number(label="Age"),

        gr.Dropdown(
            choices=["Male", "Female"],
            label="Gender"
        ),

        gr.Dropdown(
            choices=["Single", "Married", "Divorced"],
            label="Marital Status"
        ),

        gr.Dropdown(
            choices=["High School", "Bachelor", "Master", "PhD"],
            label="Education Level"
        ),

        gr.Number(label="Annual Income"),
        gr.Number(label="Monthly Income"),

        gr.Dropdown(
            choices=["Employed", "Self-Employed", "Unemployed"],
            label="Employment Status"
        ),

        gr.Number(label="Debt to Income Ratio"),
        gr.Number(label="Credit Score"),
        gr.Number(label="Loan Amount"),

        gr.Textbox(label="Loan Purpose"),

        gr.Number(label="Interest Rate"),
        gr.Number(label="Loan Term"),
        gr.Number(label="Installment"),

        gr.Textbox(label="Grade Subgrade"),

        gr.Number(label="Open Accounts"),
        gr.Number(label="Total Credit Limit"),
        gr.Number(label="Current Balance"),

        gr.Number(label="Delinquency History"),
        gr.Number(label="Public Records"),
        gr.Number(label="Number of Delinquencies"),
    ],

    outputs="text",

    title="Loan Repayment Prediction System",

    description="Predict whether a borrower will repay the loan."
)

demo.launch()

# Summary & Conclusions

## Project Overview
This notebook implements a **Loan Approval Prediction System** using 5 machine learning models trained on applicant demographic and financial data.

## Pipeline
1. **Data Loading** — Dynamic path resolution, no hardcoded paths
2. **EDA** — Class balance, negative values, IQR outliers, correlation heatmap, feature distributions
3. **Preprocessing** — Column cleanup, feature engineering (`income_loan_ratio`, `total_assets`), explicit label encoding with assertions
4. **Scaling** — `StandardScaler` applied uniformly to all models (fit on train only)
5. **Modeling** — 5 models with `GridSearchCV` hyperparameter tuning:
   - Decision Tree, Logistic Regression, Random Forest, XGBoost, KNN
6. **Evaluation** — 7 metrics (Accuracy, Precision, Recall, F1, AUC-ROC, CV mean, CV std), confusion matrices, ROC curves, feature importance
7. **Model Selection** — Best model chosen by cross-validation accuracy (not test accuracy)
8. **Explainability** — SHAP (global bar + beeswarm + local waterfall) and LIME (local feature contributions)
9. **Deployment** — Tkinter GUI with real SHAP-powered explanations per prediction

## Key Findings
- **Best Model**: Dynamically selected based on CV accuracy
- **Most Important Features**: Identified via SHAP — check the beeswarm plot above for the actual ranking on your dataset run
- **Class Balance**: Dataset distribution analyzed and stratified splitting ensures representative train/test sets
- **Data Quality**: Negative asset values detected and reported; outliers quantified via IQR

## Reproducibility
- `random_state=42` used across all splits, models, and cross-validation
- All models saved to `.pkl` files for deployment
- SHAP and LIME explainers persisted for GUI usage

## Files Produced
| File | Contents |
|---|---|
| `loan_model.pkl` | Best-performing model (selected by CV accuracy) |
| `scaler.pkl` | Fitted StandardScaler |
| `best_model_name.pkl` | Name string of the selected model |
| `shap_explainer.pkl` | SHAP explainer for the best model |
| `lime_explainer.pkl` | LIME explainer for local explanations |